## LightGBM 训练流程

本 notebook 只负责机器学习训练与推理，不调用任何 GEE函数；所有计算均在本地 Python / pandas / scikit-learn / LightGBM 中完成。

流程如下：

1. 导入依赖
2. 配置 CSV 路径、标签和特征组
3. 定义辅助函数
4. 读取正负样本 CSV 并检查数据
5. 划分训练/测试集并做基础特征筛选（缺失率、常数列、共线性）
6. 做预处理（缺失值填补 + One-Hot）和嵌入式特征筛选（基于随机森林重要性）
7. LightGBM 超参数搜索与训练
8. 测试集验证（指标、分类报告、混淆矩阵）
9. 特征重要性（split / gain）与模型公式说明
10. 对相同格式的新 CSV 输出正/负概率与置信度


In [1]:
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display

import lightgbm as lgb
from lightgbm import LGBMClassifier

from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier
from sklearn.feature_selection import SelectFromModel
from sklearn.impute import SimpleImputer
from sklearn.metrics import (
    accuracy_score,
    average_precision_score,
    classification_report,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
)
from sklearn.model_selection import (
    GroupShuffleSplit,
    RandomizedSearchCV,
    StratifiedGroupKFold,
)
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder

warnings.filterwarnings('ignore')
print('Imports loaded successfully.')
print('lightgbm version =', lgb.__version__)


Imports loaded successfully.
lightgbm version = 4.6.0


In [2]:
### 打印列名，确认表格中有什么列，为后续读表格做准备
POS_CSV_PATH = r'F:\\MYR\\Research_\\ZHANG\\Oiling\\results\\feature_batch_year\\2015\\pos_features_2015_combine.csv'
NEG_CSV_PATH = r'F:\\MYR\\Research_\\ZHANG\\Oiling\\results\\feature_batch_year\\2015\\neg_features_2015_combine.csv'

_pos_check = pd.read_csv(POS_CSV_PATH)
_neg_check = pd.read_csv(NEG_CSV_PATH)

print('正样本 CSV shape =', _pos_check.shape)
print('负样本 CSV shape =', _neg_check.shape)

print('\n正样本 CSV 列名 (%d 个):' % len(_pos_check.columns))
for c in _pos_check.columns:
    print(' -', c)

print('\n负样本 CSV 列名 (%d 个):' % len(_neg_check.columns))
for c in _neg_check.columns:
    print(' -', c)

print('\n两表列名是否一致 =', list(_pos_check.columns) == list(_neg_check.columns))
if list(_pos_check.columns) != list(_neg_check.columns):
    print('  仅在正样本存在:', set(_pos_check.columns) - set(_neg_check.columns))
    print('  仅在负样本存在:', set(_neg_check.columns) - set(_pos_check.columns))


正样本 CSV shape = (5642, 77)
负样本 CSV shape = (2268, 76)

正样本 CSV 列名 (77 个):
 - system:index
 - BUFF_DIST
 - Category
 - CenterX
 - CenterY
 - Date
 - ORIG_FID
 - Shape_Area
 - Shape_Leng
 - Subcategor
 - Year
 - area
 - asm
 - compactness
 - contrast
 - corr
 - csr
 - diss
 - elongation
 - entropy
 - era5_count
 - idm
 - image_count
 - imcorr1
 - imcorr2
 - inertia
 - instrument_mode
 - intensity_ratio
 - isdr
 - isri
 - local_mean_after
 - local_mean_before
 - max_obj
 - mean_wind
 - min_obj
 - mu_obj
 - mu_sce
 - oil_area_m2
 - oil_pixel_count
 - oil_ratio
 - perimeter
 - prom
 - rectangularity
 - sar_coverage_ratio
 - sar_missing_pixel_count
 - sar_time
 - savg
 - scene_id
 - scene_ids_used
 - sentropy
 - shade
 - sigma_obj
 - sigma_sce
 - status
 - status_34f
 - svar
 - total_pixel_count
 - var
 - vv_filtered_max
 - vv_filtered_mean
 - vv_filtered_min
 - vv_filtered_stdDev
 - vv_max
 - vv_mean
 - vv_min
 - vv_stdDev
 - wind_cv
 - wind_dir_cos
 - wind_dir_sin
 - wind_max
 - wind_min
 

In [3]:
# ===== 1) 路径、标签、泄漏防护规则与训练参数 =====
POS_CSV_PATH = r'F:\\MYR\\Research_\\ZHANG\\Oiling\\results\\feature_batch_year\\2015\\pos_features_2015_combine.csv'
NEG_CSV_PATH = r'F:\\MYR\\Research_\\ZHANG\\Oiling\\results\\feature_batch_year\\2015\\neg_features_2015_combine.csv'

LABEL_COL = 'label'
POS_LABEL = 1
NEG_LABEL = 0
RANDOM_STATE = 42
TEST_SIZE = 0.2
MISSING_THRESHOLD = 0.40
CORR_THRESHOLD = 0.95
N_ITER_SEARCH = 20
CV_FOLDS = 5

# 按场景分组，确保同一个 SAR 场景不同时进入训练/测试数据或 CV 的不同折。
GROUP_COL = 'scene_id'

# 这些列直接泄漏标签、属于 ID/元数据，或只在获得油膜结果以后才可能得到，禁止作为模型输入。
EXCLUDED_FEATURE_COLUMNS = {
    LABEL_COL,
    'system:index', '.geo',
    'scene_id', 'scene_ids_used', 'id', 'ORIG_FID',
    'Date', 'sar_time',
    'status', 'status_34f', 'sample_source',
    # 直接类别标签泄漏。
    'Category', 'Subcategor', 'Subcategory',
    # 正负样本构建过程中使用的缓冲距离，正/负样本的值完全不同。
    'BUFF_DIST',
}

# 严格模型额外排除：可能反映样本构造/后验结果的对象规模与几何字段。
STRICT_RISK_FEATURES = {
    'Shape_Leng', 'Shape_Area', 'area', 'perimeter', 'oil_area_m2',
    'oil_pixel_count', 'total_pixel_count', 'oil_ratio',
}

# CSV 中的实际坐标字段；用于分别训练“含坐标”和“不含坐标”两个变体。
GEO_FEATURES = ['CenterX', 'CenterY', 'xcoor', 'ycoor']

# 打分输出中保留的元数据列。
META_COLUMNS = ['scene_id', 'scene_ids_used', 'sar_time', 'status', 'instrument_mode', 'sample_source']

import re

def extract_year_tag(*csv_paths, default='unknown'):
    """从 CSV 文件名中提取 4 位年份（如 pos_features_pos2014.csv -> '2014'）。"""
    years_found = []
    for p in csv_paths:
        name = Path(p).stem
        matches = [m.group(0) for m in re.finditer(r'(?:19|20)\d{2}', name)]
        if matches:
            years_found.append(matches[-1])
    if not years_found:
        return default
    unique_years = sorted(set(years_found))
    if len(unique_years) > 1:
        print(f'警告：正负样本 CSV 文件名中年份不一致 {unique_years}，将使用第一个: {unique_years[0]}')
    return unique_years[0]

YEAR_TAG = extract_year_tag(POS_CSV_PATH, NEG_CSV_PATH)

print('分割策略 = 以', GROUP_COL, '为组的 GroupShuffleSplit / StratifiedGroupKFold（防止同场景泄漏）')
print('Positive CSV =', POS_CSV_PATH)
print('Negative CSV =', NEG_CSV_PATH)
print('YEAR_TAG =', YEAR_TAG)


分割策略 = 以 scene_id 为组的 GroupShuffleSplit / StratifiedGroupKFold（防止同场景泄漏）
Positive CSV = F:\\MYR\\Research_\\ZHANG\\Oiling\\results\\feature_batch_year\\2015\\pos_features_2015_combine.csv
Negative CSV = F:\\MYR\\Research_\\ZHANG\\Oiling\\results\\feature_batch_year\\2015\\neg_features_2015_combine.csv
YEAR_TAG = 2015


In [4]:
# ===== 2) 辅助函数 =====
def read_two_class_csv(pos_csv_path, neg_csv_path, label_col='label'):
    pos_path = Path(pos_csv_path)
    neg_path = Path(neg_csv_path)

    if not pos_path.exists():
        raise FileNotFoundError(f'Positive CSV not found: {pos_path}')
    if not neg_path.exists():
        raise FileNotFoundError(f'Negative CSV not found: {neg_path}')

    pos_df = pd.read_csv(pos_path).copy()
    neg_df = pd.read_csv(neg_path).copy()
    pos_df[label_col] = POS_LABEL
    neg_df[label_col] = NEG_LABEL
    pos_df['sample_source'] = 'positive_csv'
    neg_df['sample_source'] = 'negative_csv'

    df = pd.concat([pos_df, neg_df], ignore_index=True)
    return df, pos_df, neg_df


def normalize_boolean_like(series):
    if series.dtype == bool:
        return series.astype(int)

    text = series.astype(str).str.strip().str.lower()
    mapped = text.map({
        'true': 1, 'false': 0,
        '1': 1, '0': 0,
        'yes': 1, 'no': 0,
        'y': 1, 'n': 0
    })
    return mapped if mapped.notna().sum() > 0 else series


def add_time_features(df):
    df = df.copy()
    if 'sar_time' in df.columns:
        sar_dt = pd.to_datetime(df['sar_time'], errors='coerce')
        df['sar_year'] = sar_dt.dt.year
        df['sar_month'] = sar_dt.dt.month
        df['sar_day'] = sar_dt.dt.day
        df['sar_hour'] = sar_dt.dt.hour
    if 'wind_valid' in df.columns:
        df['wind_valid'] = normalize_boolean_like(df['wind_valid'])
    return df


def build_candidate_feature_list(df):
    """使用 CSV 实际列：排除标签、标识、原始时间/几何及状态泄漏字段。"""
    return [c for c in df.columns if c not in EXCLUDED_FEATURE_COLUMNS]


def drop_high_missing_columns(X_train, X_test, threshold=0.4):
    missing_ratio = X_train.isna().mean()
    drop_cols = missing_ratio[missing_ratio > threshold].index.tolist()
    return X_train.drop(columns=drop_cols), X_test.drop(columns=drop_cols), drop_cols


def drop_constant_columns(X_train, X_test):
    drop_cols = [c for c in X_train.columns if X_train[c].nunique(dropna=False) <= 1]
    return X_train.drop(columns=drop_cols), X_test.drop(columns=drop_cols), drop_cols


def drop_high_corr_numeric_columns(X_train, X_test, threshold=0.95):
    numeric_cols = X_train.select_dtypes(include=[np.number, bool]).columns.tolist()
    if len(numeric_cols) <= 1:
        return X_train, X_test, []

    corr = X_train[numeric_cols].corr().abs()
    upper = corr.where(np.triu(np.ones(corr.shape), k=1).astype(bool))
    drop_cols = [col for col in upper.columns if (upper[col] > threshold).any()]
    return X_train.drop(columns=drop_cols), X_test.drop(columns=drop_cols), drop_cols


def split_column_types(df):
    numeric_cols = df.select_dtypes(include=[np.number, bool]).columns.tolist()
    categorical_cols = [c for c in df.columns if c not in numeric_cols]
    return numeric_cols, categorical_cols


def build_preprocessor(X_train):
    numeric_cols, categorical_cols = split_column_types(X_train)
    transformers = []

    if numeric_cols:
        transformers.append(('num', SimpleImputer(strategy='median'), numeric_cols))
    if categorical_cols:
        transformers.append((
            'cat',
            Pipeline([
                ('imputer', SimpleImputer(strategy='most_frequent')),
                ('onehot', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
            ]),
            categorical_cols
        ))

    preprocessor = ColumnTransformer(
        transformers=transformers,
        remainder='drop',
        verbose_feature_names_out=False
    )
    return preprocessor, numeric_cols, categorical_cols


def collect_metrics(y_true, y_pred, y_proba_pos):
    return {
        'accuracy': accuracy_score(y_true, y_pred),
        'precision': precision_score(y_true, y_pred, zero_division=0),
        'recall': recall_score(y_true, y_pred, zero_division=0),
        'f1': f1_score(y_true, y_pred, zero_division=0),
        'roc_auc': roc_auc_score(y_true, y_proba_pos),
        'pr_auc': average_precision_score(y_true, y_proba_pos),
    }

print('Helper functions ready.')

Helper functions ready.


In [5]:
# ===== 3) 读取 CSV 并检查数据 =====
full_df, pos_df_raw, neg_df_raw = read_two_class_csv(POS_CSV_PATH, NEG_CSV_PATH, LABEL_COL)
full_df = add_time_features(full_df)

feature_candidates = build_candidate_feature_list(full_df)
excluded_columns_present = [c for c in full_df.columns if c in EXCLUDED_FEATURE_COLUMNS]
if len(feature_candidates) == 0:
    raise ValueError('没有找到可用特征列，请检查 CSV 字段名和排除规则。')

print('positive sample count =', len(pos_df_raw))
print('negative sample count =', len(neg_df_raw))
print('total sample count =', len(full_df))
print('candidate feature count =', len(feature_candidates))

print('label distribution =')
display(full_df[LABEL_COL].value_counts(dropna=False).sort_index())

print('excluded columns =')
display(pd.Series(excluded_columns_present, name='excluded_column'))

print('candidate features =')
display(pd.Series(feature_candidates, name='feature'))

full_df.head(3)

positive sample count = 5642
negative sample count = 2268
total sample count = 7910
candidate feature count = 69
label distribution =


label
0    2268
1    5642
Name: count, dtype: int64

excluded columns =


0       system:index
1          BUFF_DIST
2           Category
3               Date
4           ORIG_FID
5         Subcategor
6           sar_time
7           scene_id
8     scene_ids_used
9             status
10        status_34f
11              .geo
12             label
13     sample_source
Name: excluded_column, dtype: object

candidate features =


0        CenterX
1        CenterY
2     Shape_Area
3     Shape_Leng
4           Year
         ...    
64         ycoor
65      sar_year
66     sar_month
67       sar_day
68      sar_hour
Name: feature, Length: 69, dtype: object

,system:index,BUFF_DIST,Category,CenterX,CenterY,Date,ORIG_FID,Shape_Area,Shape_Leng,Subcategor,...,wind_valid,xcoor,ycoor,.geo,label,sample_source,sar_year,sar_month,sar_day,sar_hour
0,00000000000000000004,3000.0,Positive,41.1081,41.9787,20151121,3477,0.003909,0.252781,Natural seeps,...,1,41.105449,41.974995,"{""type"":""Polygon"",""coordinates"":[[[41.10810031...",1,positive_csv,2015,11,21,3
1,00000000000000000010,3000.0,Positive,20.8651,37.6176,20151123,3489,0.003673,0.243996,Natural seeps,...,1,20.864369,37.614228,"{""type"":""Polygon"",""coordinates"":[[[20.86510149...",1,positive_csv,2015,11,23,4
2,00000000000000000011,3000.0,Positive,20.8029,37.1587,20151123,3490,0.003651,0.243178,Natural seeps,...,1,20.800968,37.157201,"{""type"":""Polygon"",""coordinates"":[[[20.80289922...",1,positive_csv,2015,11,23,4


In [6]:
# ===== 4) 按场景分组划分训练/测试集（防止同场景泄漏） =====
if GROUP_COL not in full_df.columns:
    raise KeyError(f'未找到分组列 {GROUP_COL!r}，无法进行防泄漏的场景分组划分。')

groups_all = full_df[GROUP_COL].fillna('__MISSING_SCENE__').astype(str)
if groups_all.nunique() < 2:
    raise ValueError(f'{GROUP_COL!r} 的有效分组不足 2 个，无法划分训练/测试集。')

# GroupShuffleSplit 保证任何一个 scene_id 只会归入训练集或测试集的一侧。
gss = GroupShuffleSplit(n_splits=1, test_size=TEST_SIZE, random_state=RANDOM_STATE)
train_pos, test_pos = next(gss.split(full_df, full_df[LABEL_COL], groups=groups_all))

train_df = full_df.iloc[train_pos].copy()
test_df = full_df.iloc[test_pos].copy()
y_train = train_df[LABEL_COL].astype(int)
y_test = test_df[LABEL_COL].astype(int)
groups_train = groups_all.iloc[train_pos]
groups_test = groups_all.iloc[test_pos]

scene_overlap = set(groups_train) & set(groups_test)
if scene_overlap:
    raise RuntimeError(f'场景分割失败：训练/测试仍有 {len(scene_overlap)} 个重叠场景。')
if y_train.nunique() < 2 or y_test.nunique() < 2:
    raise ValueError('场景分组切分后某一侧只剩单一类别；请更改 RANDOM_STATE 或 TEST_SIZE。')

print('train size =', len(train_df), '| test size =', len(test_df))
print('train scene count =', groups_train.nunique(), '| test scene count =', groups_test.nunique())
print('train/test scene overlap =', len(scene_overlap), '(必须为 0)')
print('train label distribution =')
display(y_train.value_counts().sort_index())
print('test label distribution =')
display(y_test.value_counts().sort_index())


train size = 6509 | test size = 1401
train scene count = 1770 | test scene count = 443
train/test scene overlap = 0 (必须为 0)
train label distribution =


label
0    1875
1    4634
Name: count, dtype: int64

test label distribution =


label
0     393
1    1008
Name: count, dtype: int64

In [7]:
# ===== 5) 定义三个特征集：含坐标 / 不含坐标 / 严格防泄漏 =====
feature_candidates_geo = list(feature_candidates)
feature_candidates_nogeo = [c for c in feature_candidates if c not in GEO_FEATURES]

# 严格模型：不使用坐标，也不使用可能反映样本构造、缓冲或后验结果的高风险尺度/几何字段。
feature_candidates_strict = [
    c for c in feature_candidates
    if c not in GEO_FEATURES and c not in STRICT_RISK_FEATURES
]

present_geo_features = [c for c in GEO_FEATURES if c in feature_candidates]
present_strict_risk_features = [c for c in STRICT_RISK_FEATURES if c in feature_candidates]

print('含坐标特征候选数 =', len(feature_candidates_geo))
print('不含坐标特征候选数 =', len(feature_candidates_nogeo))
print('严格特征候选数 =', len(feature_candidates_strict))
print('不含坐标模型剔除的坐标字段 =', present_geo_features)
print('严格模型额外剔除的高风险字段 =', present_strict_risk_features)


含坐标特征候选数 = 69
不含坐标特征候选数 = 65
严格特征候选数 = 57
不含坐标模型剔除的坐标字段 = ['CenterX', 'CenterY', 'xcoor', 'ycoor']
严格模型额外剔除的高风险字段 = ['oil_ratio', 'oil_pixel_count', 'perimeter', 'oil_area_m2', 'total_pixel_count', 'Shape_Area', 'area', 'Shape_Leng']


In [8]:
# ===== 6) 通用训练函数：特征筛选 + 按场景分组 CV + LightGBM =====
def train_lgbm_variant(feature_list, variant_name, model_suffix, year_tag=YEAR_TAG):
    print(f"\n{'=' * 18} {variant_name} | year={year_tag} {'=' * 18}")
    X_tr = train_df[feature_list].copy()
    X_te = test_df[feature_list].copy()

    # 基础筛选仅依据训练集进行，测试集只同步应用相同的列删除规则。
    X_tr, X_te, dropped_missing = drop_high_missing_columns(X_tr, X_te, MISSING_THRESHOLD)
    X_tr, X_te, dropped_constant = drop_constant_columns(X_tr, X_te)
    X_tr, X_te, dropped_corr = drop_high_corr_numeric_columns(X_tr, X_te, CORR_THRESHOLD)
    kept_cols = X_tr.columns.tolist()
    if not kept_cols:
        raise ValueError(f'{variant_name}: 基础筛选后无可用特征。')

    # Pipeline 让每个 CV 折独立拟合：预处理、嵌入式筛选和模型均不会看到该折验证数据。
    preproc, _, _ = build_preprocessor(X_tr)
    selector_rf = RandomForestClassifier(
        n_estimators=300, class_weight='balanced', random_state=RANDOM_STATE,
        n_jobs=-1, min_samples_leaf=2
    )
    pipeline = Pipeline([
        ('preprocessor', preproc),
        ('selector', SelectFromModel(selector_rf, threshold='median')),
        ('model', LGBMClassifier(
            objective='binary', class_weight='balanced', random_state=RANDOM_STATE,
            n_jobs=-1, verbose=-1
        )),
    ])

    # 每个 CV 折按 scene_id 分组，同场景绝不跨越拟合/验证两侧。
    cv = StratifiedGroupKFold(n_splits=CV_FOLDS, shuffle=True, random_state=RANDOM_STATE)
    cv_splits = list(cv.split(X_tr, y_train, groups=groups_train))
    for fold, (fit_i, val_i) in enumerate(cv_splits, 1):
        overlap = set(groups_train.iloc[fit_i]) & set(groups_train.iloc[val_i])
        if overlap:
            raise RuntimeError(f'CV fold {fold} 存在 {len(overlap)} 个场景重叠。')

    param_distributions = {
        'model__n_estimators': [100, 200, 300, 500, 800],
        'model__num_leaves': [7, 15, 31, 63, 127],
        'model__max_depth': [-1, 3, 5, 7, 9],
        'model__learning_rate': [0.01, 0.03, 0.05, 0.08, 0.1],
        'model__min_child_samples': [5, 10, 20, 30, 50],
        'model__subsample': [0.6, 0.7, 0.8, 0.9, 1.0],
        'model__colsample_bytree': [0.6, 0.7, 0.8, 0.9, 1.0],
        'model__reg_alpha': [0.0, 0.01, 0.1, 1.0],
        'model__reg_lambda': [0.0, 0.01, 0.1, 1.0, 5.0],
    }
    search = RandomizedSearchCV(
        pipeline, param_distributions=param_distributions, n_iter=N_ITER_SEARCH,
        scoring='f1', cv=cv_splits, random_state=RANDOM_STATE, n_jobs=-1,
        verbose=0, refit=True
    ).fit(X_tr, y_train)
    fitted_pipeline = search.best_estimator_

    # 从在完整训练集上 refit 的 pipeline 提取模型与实际入模特征。
    fitted_preproc = fitted_pipeline.named_steps['preprocessor']
    fitted_selector = fitted_pipeline.named_steps['selector']
    model = fitted_pipeline.named_steps['model']
    pre_names = fitted_preproc.get_feature_names_out()
    selected_mask = fitted_selector.get_support()
    selected_names = list(np.asarray(pre_names)[selected_mask])

    y_pred = fitted_pipeline.predict(X_te)
    y_proba_pos = fitted_pipeline.predict_proba(X_te)[:, 1]
    metrics = collect_metrics(y_test, y_pred, y_proba_pos)
    booster = model.booster_
    importance = pd.DataFrame({
        'feature': selected_names,
        'importance_split': booster.feature_importance(importance_type='split'),
        'importance_gain': booster.feature_importance(importance_type='gain'),
    })
    importance['gain_ratio'] = importance['importance_gain'] / importance['importance_gain'].sum()
    importance = importance.sort_values('importance_gain', ascending=False).reset_index(drop=True)

    model_path = Path(f'lgbm_oil_model{model_suffix}_{year_tag}.txt')
    booster.save_model(str(model_path))
    clean_params = {k.replace('model__', ''): v for k, v in search.best_params_.items()}
    print('基础筛选后特征数 =', len(kept_cols), '| 嵌入式筛选后特征数 =', len(selected_names))
    print('分组 CV F1 =', search.best_score_)
    print('最优参数 =', clean_params)
    print('独立场景测试指标 =')
    display(pd.Series(metrics))

    return {
        'variant_name': variant_name, 'model_suffix': model_suffix, 'year_tag': year_tag,
        'feature_candidates': feature_list, 'kept_feature_columns': kept_cols,
        'selected_feature_names': selected_names, 'preprocessor': fitted_preproc,
        'selector': fitted_selector, 'selected_mask': selected_mask,
        'pipeline': fitted_pipeline, 'model': model, 'booster': booster,
        'metrics': metrics, 'feature_importance': importance,
        'best_params': clean_params, 'best_cv_f1': search.best_score_,
        'model_text_path': model_path,
    }

print('已定义严格的按 scene_id 分组训练函数：CV 内部独立完成预处理和特征筛选。')


已定义严格的按 scene_id 分组训练函数：CV 内部独立完成预处理和特征筛选。


In [9]:
# ===== 7a) 模型 A：含坐标特征 =====
lgbm_artifacts = train_lgbm_variant(
    feature_candidates_geo,
    variant_name='含坐标 (with_geo)',
    model_suffix='',
    year_tag=YEAR_TAG
)



================== 含坐标 (with_geo) | year=2015 ==================


基础筛选后特征数 = 38 | 嵌入式筛选后特征数 = 20
分组 CV F1 = 1.0
最优参数 = {'subsample': 0.6, 'reg_lambda': 0.01, 'reg_alpha': 0.1, 'num_leaves': 15, 'n_estimators': 300, 'min_child_samples': 30, 'max_depth': 7, 'learning_rate': 0.01, 'colsample_bytree': 0.8}
独立场景测试指标 =


accuracy     0.997145
precision    0.996047
recall       1.000000
f1           0.998020
roc_auc      0.999849
pr_auc       0.999940
dtype: float64

In [10]:
# ===== 7b) 模型 B：不含坐标特征（后缀 _nogeo） =====
lgbm_artifacts_nogeo = train_lgbm_variant(
    feature_candidates_nogeo,
    variant_name='不含坐标 (no_geo)',
    model_suffix='_nogeo',
    year_tag=YEAR_TAG
)



================== 不含坐标 (no_geo) | year=2015 ==================


基础筛选后特征数 = 36 | 嵌入式筛选后特征数 = 19
分组 CV F1 = 1.0
最优参数 = {'subsample': 0.6, 'reg_lambda': 0.01, 'reg_alpha': 0.1, 'num_leaves': 127, 'n_estimators': 100, 'min_child_samples': 30, 'max_depth': 9, 'learning_rate': 0.03, 'colsample_bytree': 1.0}
独立场景测试指标 =


accuracy     0.997145
precision    0.996047
recall       1.000000
f1           0.998020
roc_auc      0.999662
pr_auc       0.999825
dtype: float64

In [11]:
# ===== 7c) 模型 C：严格特征模型（无坐标 + 无高风险几何/后验字段，后缀 _strict） =====
lgbm_artifacts_strict = train_lgbm_variant(
    feature_candidates_strict,
    variant_name='严格特征 (no_geo_strict)',
    model_suffix='_strict',
    year_tag=YEAR_TAG
)



================== 严格特征 (no_geo_strict) | year=2015 ==================


基础筛选后特征数 = 32 | 嵌入式筛选后特征数 = 17
分组 CV F1 = 0.9188761886963208
最优参数 = {'subsample': 0.8, 'reg_lambda': 0.01, 'reg_alpha': 0.01, 'num_leaves': 127, 'n_estimators': 500, 'min_child_samples': 20, 'max_depth': -1, 'learning_rate': 0.05, 'colsample_bytree': 0.6}
独立场景测试指标 =


accuracy     0.882227
precision    0.899526
recall       0.941468
f1           0.920019
roc_auc      0.929945
pr_auc       0.965978
dtype: float64

In [12]:
# ===== 8) 特征重要性与模型公式说明（三个变体） =====
def print_formula_explanation(artifacts):
    model_ = artifacts['model']
    booster_ = artifacts['booster']
    print(f"\n--- {artifacts['variant_name']} ---")
    print('LightGBM 预测：F(x) = base_score + Σ[learning_rate × tree_t(x)]')
    print('正类概率：P(oil=1|x) = 1 / (1 + exp(-F(x)))')
    print(f"树数量 = {booster_.num_trees()}，learning_rate = {model_.get_params()['learning_rate']}")
    print(f"实际使用特征（{len(artifacts['selected_feature_names'])} 个）：")
    print(artifacts['selected_feature_names'])
    print('按 gain 排序的前 10 个特征：')
    display(artifacts['feature_importance'][['feature', 'gain_ratio']].head(10))
    print('模型文件：', artifacts['model_text_path'].resolve())

for artifacts_ in [lgbm_artifacts, lgbm_artifacts_nogeo, lgbm_artifacts_strict]:
    print_formula_explanation(artifacts_)



--- 含坐标 (with_geo) ---
LightGBM 预测：F(x) = base_score + Σ[learning_rate × tree_t(x)]
正类概率：P(oil=1|x) = 1 / (1 + exp(-F(x)))
树数量 = 300，learning_rate = 0.01
实际使用特征（20 个）：
['CenterX', 'CenterY', 'Shape_Area', 'area', 'asm', 'compactness', 'corr', 'idm', 'intensity_ratio', 'isdr', 'local_mean_after', 'mean_wind', 'perimeter', 'rectangularity', 'savg', 'shade', 'total_pixel_count', 'vv_filtered_max', 'vv_filtered_stdDev', 'sar_hour']
按 gain 排序的前 10 个特征：


,feature,gain_ratio
0,Shape_Area,0.544463
1,CenterY,0.236705
2,total_pixel_count,0.098001
3,CenterX,0.045873
4,vv_filtered_stdDev,0.023123
5,area,0.016465
6,compactness,0.015487
7,perimeter,0.005636
8,mean_wind,0.003798
9,local_mean_after,0.002913


模型文件： F:\jupyter\Slick2Vessel\notebooks\lgbm_oil_model_2015.txt

--- 不含坐标 (no_geo) ---
LightGBM 预测：F(x) = base_score + Σ[learning_rate × tree_t(x)]
正类概率：P(oil=1|x) = 1 / (1 + exp(-F(x)))
树数量 = 100，learning_rate = 0.03
实际使用特征（19 个）：
['Shape_Area', 'area', 'asm', 'compactness', 'corr', 'diss', 'idm', 'intensity_ratio', 'isdr', 'local_mean_after', 'mean_wind', 'perimeter', 'rectangularity', 'savg', 'shade', 'total_pixel_count', 'vv_filtered_max', 'vv_filtered_stdDev', 'sar_hour']
按 gain 排序的前 10 个特征：


,feature,gain_ratio
0,Shape_Area,0.605786
1,total_pixel_count,0.394214
2,mean_wind,0.000000
3,vv_filtered_stdDev,0.000000
4,vv_filtered_max,0.000000
5,shade,0.000000
6,savg,0.000000
7,rectangularity,0.000000
8,perimeter,0.000000
9,local_mean_after,0.000000


模型文件： F:\jupyter\Slick2Vessel\notebooks\lgbm_oil_model_nogeo_2015.txt

--- 严格特征 (no_geo_strict) ---
LightGBM 预测：F(x) = base_score + Σ[learning_rate × tree_t(x)]
正类概率：P(oil=1|x) = 1 / (1 + exp(-F(x)))
树数量 = 500，learning_rate = 0.05
实际使用特征（17 个）：
['asm', 'compactness', 'contrast', 'corr', 'csr', 'diss', 'intensity_ratio', 'isdr', 'isri', 'local_mean_after', 'mean_wind', 'prom', 'rectangularity', 'savg', 'shade', 'vv_filtered_stdDev', 'sar_hour']
按 gain 排序的前 10 个特征：


,feature,gain_ratio
0,vv_filtered_stdDev,0.145793
1,sar_hour,0.121863
2,corr,0.081172
3,compactness,0.073035
4,local_mean_after,0.068171
5,csr,0.060769
6,mean_wind,0.059576
7,contrast,0.052334
8,shade,0.048453
9,isdr,0.047225


模型文件： F:\jupyter\Slick2Vessel\notebooks\lgbm_oil_model_strict_2015.txt


In [13]:
# ===== 9) 三个模型的防泄漏评估对比 =====
def build_summary(artifacts):
    return {
        'variant': artifacts['variant_name'],
        'model_suffix': artifacts['model_suffix'],
        'n_candidate_features': len(artifacts['feature_candidates']),
        'n_filtered_features': len(artifacts['kept_feature_columns']),
        'n_selected_features': len(artifacts['selected_feature_names']),
        'grouped_cv_f1': artifacts['best_cv_f1'],
        'test_accuracy': artifacts['metrics']['accuracy'],
        'test_precision': artifacts['metrics']['precision'],
        'test_recall': artifacts['metrics']['recall'],
        'test_f1': artifacts['metrics']['f1'],
        'test_roc_auc': artifacts['metrics']['roc_auc'],
        'test_pr_auc': artifacts['metrics']['pr_auc'],
        'model_text_path': str(artifacts['model_text_path']),
    }

comparison_table = pd.DataFrame([
    build_summary(lgbm_artifacts),
    build_summary(lgbm_artifacts_nogeo),
    build_summary(lgbm_artifacts_strict),
]).set_index('variant')

print('===== 三种模型对比（训练/测试及CV均按 scene_id 隔离） =====')
display(comparison_table.T)

metric_columns = ['grouped_cv_f1', 'test_accuracy', 'test_precision', 'test_recall', 'test_f1', 'test_roc_auc', 'test_pr_auc']
strict_vs_nogeo = comparison_table.loc['严格特征 (no_geo_strict)', metric_columns] - comparison_table.loc['不含坐标 (no_geo)', metric_columns]
print('\n严格模型 - 不含坐标模型（负值表示剔除高风险特征后性能降低）：')
display(strict_vs_nogeo)

print('\n建议：严格特征模型的独立场景测试指标是最适合作为泛化性能报告与跨新场景预测依据的结果。')


===== 三种模型对比（训练/测试及CV均按 scene_id 隔离） =====


variant,含坐标 (with_geo),不含坐标 (no_geo),严格特征 (no_geo_strict)
model_suffix,,_nogeo,_strict
n_candidate_features,69,65,57
n_filtered_features,38,36,32
n_selected_features,20,19,17
grouped_cv_f1,1.0,1.0,0.918876
test_accuracy,0.997145,0.997145,0.882227
test_precision,0.996047,0.996047,0.899526
test_recall,1.0,1.0,0.941468
test_f1,0.99802,0.99802,0.920019
test_roc_auc,0.999849,0.999662,0.929945



严格模型 - 不含坐标模型（负值表示剔除高风险特征后性能降低）：


grouped_cv_f1    -0.081124
test_accuracy    -0.114918
test_precision   -0.096521
test_recall      -0.058532
test_f1             -0.078
test_roc_auc     -0.069717
test_pr_auc      -0.033847
dtype: object


建议：严格特征模型的独立场景测试指标是最适合作为泛化性能报告与跨新场景预测依据的结果。


In [14]:
# ===== 10) 对相同格式的新 CSV 打分：三个模型变体通用 =====
def score_csv_with_artifacts(csv_path, artifacts, positive_threshold=0.5):
    """使用指定模型工件对同格式 CSV 输出负/正概率、类别和置信度。"""
    csv_path = Path(csv_path)
    if not csv_path.exists():
        raise FileNotFoundError(f'CSV not found: {csv_path}')

    new_df = add_time_features(pd.read_csv(csv_path))
    missing_cols = [c for c in artifacts['kept_feature_columns'] if c not in new_df.columns]
    for c in missing_cols:
        new_df[c] = np.nan

    X_new = new_df[artifacts['kept_feature_columns']].copy()
    # 工件中的 pipeline 已包含同训练阶段一致的预处理与特征筛选。
    proba_pos = artifacts['pipeline'].predict_proba(X_new)[:, 1]
    proba_neg = 1 - proba_pos
    pred_label = (proba_pos >= positive_threshold).astype(int)

    out_cols = [c for c in META_COLUMNS if c in new_df.columns]
    scored_df = new_df[out_cols].copy() if out_cols else pd.DataFrame(index=new_df.index)
    scored_df['pred_label'] = pred_label
    scored_df['pred_class'] = np.where(pred_label == 1, 'positive', 'negative')
    scored_df['prob_negative'] = proba_neg
    scored_df['prob_positive'] = proba_pos
    scored_df['confidence'] = np.where(pred_label == 1, proba_pos, proba_neg)
    return scored_df


def score_csv_with_lgbm(csv_path, positive_threshold=0.5):
    """含坐标模型；仅建议用于与既有区域相近的应用。"""
    return score_csv_with_artifacts(csv_path, lgbm_artifacts, positive_threshold)


def score_csv_with_lgbm_nogeo(csv_path, positive_threshold=0.5):
    """不含坐标模型，但仍包含高风险几何/后验字段。"""
    return score_csv_with_artifacts(csv_path, lgbm_artifacts_nogeo, positive_threshold)


def score_csv_with_lgbm_strict(csv_path, positive_threshold=0.5):
    """严格模型：跨新场景预测时推荐使用。"""
    return score_csv_with_artifacts(csv_path, lgbm_artifacts_strict, positive_threshold)

print('三个打分函数已就绪：')
print('  score_csv_with_lgbm(csv_path)        -> 含坐标模型')
print('  score_csv_with_lgbm_nogeo(csv_path)  -> 不含坐标模型')
print('  score_csv_with_lgbm_strict(csv_path) -> 严格模型（推荐用于新场景）')


三个打分函数已就绪：
  score_csv_with_lgbm(csv_path)        -> 含坐标模型
  score_csv_with_lgbm_nogeo(csv_path)  -> 不含坐标模型
  score_csv_with_lgbm_strict(csv_path) -> 严格模型（推荐用于新场景）


In [15]:
# ===== 自测：三个模型在负样本 CSV 上的打分差异 =====
scores_geo = score_csv_with_lgbm(NEG_CSV_PATH)
scores_nogeo = score_csv_with_lgbm_nogeo(NEG_CSV_PATH)
scores_strict = score_csv_with_lgbm_strict(NEG_CSV_PATH)

score_rates = pd.Series({
    '含坐标模型_预测正类比例': scores_geo['pred_label'].mean(),
    '不含坐标模型_预测正类比例': scores_nogeo['pred_label'].mean(),
    '严格模型_预测正类比例': scores_strict['pred_label'].mean(),
})
print('负样本 CSV 自测：')
display(score_rates)

compare_preview = pd.DataFrame({
    'prob_positive_geo': scores_geo['prob_positive'].to_numpy(),
    'prob_positive_nogeo': scores_nogeo['prob_positive'].to_numpy(),
    'prob_positive_strict': scores_strict['prob_positive'].to_numpy(),
    'pred_geo': scores_geo['pred_label'].to_numpy(),
    'pred_nogeo': scores_nogeo['pred_label'].to_numpy(),
    'pred_strict': scores_strict['pred_label'].to_numpy(),
})
display(compare_preview.head(10))


负样本 CSV 自测：


含坐标模型_预测正类比例     0.001764
不含坐标模型_预测正类比例    0.001764
严格模型_预测正类比例      0.046737
dtype: float64

,prob_positive_geo,prob_positive_nogeo,prob_positive_strict,pred_geo,pred_nogeo,pred_strict
0,0.097855,0.024635,0.999989,0,0,1
1,0.082126,0.025822,0.992514,0,0,1
2,0.072991,0.024635,0.002160,0,0,0
3,0.074665,0.029521,0.000785,0,0,0
4,0.066218,0.029521,0.000267,0,0,0
5,0.115889,0.058851,0.002159,0,0,0
6,0.084089,0.029521,0.000440,0,0,0
7,0.090562,0.029521,0.000544,0,0,0
8,0.075740,0.029521,0.001331,0,0,0
9,0.084996,0.029521,0.000959,0,0,0


In [ ]:
#######预测模块，输入region分割得到的特征#########
